### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  # --multirun

In [ ]:
Quick CLI Override (no file edit)
# Use LogSpectrogram (original)
python -m emg2qwerty.train \
  user="single_user" \
  transforms.logspec._target_=emg2qwerty.transforms.LogSpectrogram \
  module.in_features=528
# Use NewLogSpectrogram (RSG)
python -m emg2qwerty.train \
  user="single_user" \
  transforms.logspec._target_=emg2qwerty.transforms.NewLogSpectrogram \
  module.in_features=96

In [ ]:
Enable transformer:
python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  module.use_transformer=true
Customize transformer parameters:
python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  module.use_transformer=true \
  module.transformer_d_model=512 \
  module.transformer_num_layers=4

The environment isn't set up for testing, but the code should work. Let me summarize what I've added:
Added Transformer Block with Configurable Enable/Disable
Changes Made:
1. modules.py - Added Transformer components:
   - PositionalEncoding - Adds positional information to sequences
   - TransformerEncoderBlock - Single transformer encoder layer with self-attention
   - TransformerEncoder - Full transformer encoder with multiple layers
2. lightning.py - Updated TDSConvCTCModule to optionally include transformer:
   - Added use_transformer parameter (default: False)
   - When enabled, adds linear projection → transformer → linear projection back
   - Maintains compatibility when disabled
3. config/model/tds_conv_ctc.yaml - Added transformer configuration options:
      use_transformer: false  # Enable/disable transformer block
   transformer_d_model: 256
   transformer_nhead: 8
   transformer_num_layers: 6
   transformer_dim_feedforward: 1024
   
Usage:
Enable transformer:
python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  module.use_transformer=true
Customize transformer parameters:
python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu \
  module.use_transformer=true \
  module.transformer_d_model=512 \
  module.transformer_num_layers=4
Keep transformer disabled (default):
python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="/home/ubuntu/jinru/emg2qwerty/logs/2026-02-27/07-54-39/checkpoints/last.ckpt" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  --multirun